# 3.0 모델링
2.0에서 만든 연간 정답 표로 다음 해 잘버팀을 예측한다.
- 모델 고르기: 결과연도 2023 학습 → 2024 검증
- 최종 시험: 2023~2024 학습 → 2025 (딱 한 번)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cafe_target_annual_2022_2025.csv")
print(df.shape)

(3504, 30)


## 1. 입력 칸 정하기와 연도별 나누기 ✅
입력: 작년 변화율, 작년에 카페가 유동인구보다 덜 늘었나, 폐업률, 상권 유형 (2.0 EDA 결론의 후보)
- 빈칸이 있는 행을 빼서 결과연도별 2023: 863행, 2024: 874행, 2025: 865행 (2022는 작년 값이 없어 전부 빠짐)
- 학습 2023 / 검증 2024 / 최종 시험 2025 (시험 연도는 마지막에 한 번만 봄)
- 정답 칸(다음 해 매출, 변화율, 중앙값, 잘버팀)과 상권 코드·이름은 입력에서 뺌

In [2]:
# H1 예측 버전 칸 다시 만들기 (2.0에서는 탐색용 표에만 만들었음)
df["작년_카페가_덜_늘었나"] = (df["작년_카페_증가율"] < df["작년_유동인구_증가율"]).astype(int)
df.loc[df["작년_카페_증가율"].isna() | df["작년_유동인구_증가율"].isna(), "작년_카페가_덜_늘었나"] = np.nan

입력칸 = ["작년_변화율", "작년_카페가_덜_늘었나", "폐업률", "상권_구분_코드_명"]   # ★ 모델이 보는 칸
쓸행 = df.dropna(subset=입력칸).copy()
print(쓸행.groupby("결과연도").size())

train = 쓸행[쓸행["결과연도"] == 2023]     # 학습
val = 쓸행[쓸행["결과연도"] == 2024]       # 검증 (모델 고르기)
test = 쓸행[쓸행["결과연도"] == 2025]      # ★ 최종 시험: 마지막까지 열어 보지 않음
print(train.shape, val.shape)
print("검증 연도 잘버팀 비율:", round(val["잘버팀"].mean(), 3))

결과연도
2023    863
2024    874
2025    865
dtype: int64
(863, 31) (874, 31)
검증 연도 잘버팀 비율: 0.503


## 2. 기준선 ✅
- 동전 던지기: 약 50% (검증 연도 실제 잘버팀 비율 50.3%)
- 작년 변화율 규칙(가운데보다 높으면 1): 정확도 55.0%
- 작년 변화율 상위 20%를 뽑으면 62.3%가 실제로 잘 버팀 → 모델은 이 62.3%를 넘어야 의미가 있음
- 주 점수는 상위 20% 적중률: 추천받은 상권이 실제로 괜찮은지가 목적에 더 맞음

In [3]:
# 상위 20% 적중률: 점수가 높은 20% 상권 중 실제로 잘 버틴 비율
def 상위20(점수, 정답):
    기준 = 점수.quantile(0.8)              # 위에서 20% 지점 값
    return 정답[점수 >= 기준].mean()

# 기준선: 작년 변화율이 가운데보다 높으면 '잘버팀'이라고 찍기
규칙 = (val["작년_변화율"] > val["작년_변화율"].median()).astype(int)
print("작년 변화율 규칙 - 정확도:", round((규칙 == val["잘버팀"]).mean(), 3))
print("작년 변화율 규칙 - 상위 20% 적중률:", round(상위20(val["작년_변화율"], val["잘버팀"]), 3))

작년 변화율 규칙 - 정확도: 0.55
작년 변화율 규칙 - 상위 20% 적중률: 0.623
